## Install Ollama & Pyngrok

In [ ]:
# Ignore non-critical repo update errors (e.g. CRAN mirror sync issues) and install zstd
!apt-get update || true
!apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install Python dependencies
!pip install pyngrok discord.py nest-asyncio requests

## Start Ollama Server with CORS Enabled

In [ ]:
import os
import subprocess
import time

# Enable cross-origin requests
os.environ['OLLAMA_ORIGINS'] = '*'

# Allow up to 2 models to be loaded into VRAM simultaneously (llama3.1:8b + llama3.2-vision:11b)
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '2'

# Enable parallel requests per model (optional, defaults to 4 in modern Ollama)
os.environ['OLLAMA_NUM_PARALLEL'] = '2'

# Enable deep logging to see incoming payloads, prompts, and generation metrics
os.environ['OLLAMA_DEBUG'] = '1'
os.environ['OLLAMA_DEBUG_LOG_REQUESTS'] = '1'

# Start Ollama service in background and route all output to ollama.log
print("Starting Ollama server with multi-model support and debug logs enabled...")
log_file = open("ollama.log", "w")
subprocess.Popen(['ollama', 'serve'], stdout=log_file, stderr=subprocess.STDOUT)

# Give the server a few seconds to fully boot up
time.sleep(3)
print("Server is running! Traffic is being recorded.")

## Pull Llama 3.1 8B Model

In [ ]:
# Pull Text Model (used by api/chat.js)
# !ollama pull llama3.1:8b
!ollama pull Hudson/llama3.1-uncensored:8b

# Pull Vision Model (used by api/vision.js)
!ollama pull llama3.2-vision:11b

## Pre-warm

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"

models = ["Hudson/llama3.1-uncensored:8b", "llama3.2-vision:11b"]

for model in models:
    print(f"Pre-warming model: {model}...")
    try:
        res = requests.post(OLLAMA_URL, json={"model": model, "keep_alive": -1})
        if res.status_code == 200:
            print(f"Successfully loaded {model} into VRAM.")
    except Exception as e:
        print(f"Failed to pre-warm {model}: {e}")

## Expose Port 11434 with ngrok Tunnel

In [ ]:
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

# Get your free authtoken from Kaggle Secrets
user_secrets = UserSecretsClient()
NGROK_AUTH_TOKEN = user_secrets.get_secret('NGROK')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Expose Ollama's default port AND rewrite the host header to bypass the 403 error
# Force ngrok to use the IPv4 loopback address specifically
tunnel = ngrok.connect(
    "127.0.0.1:11434", 
    host_header="localhost:11434"
)
public_url = tunnel.public_url

print(f"\n=======================================================")
print(f"  Copy this base URL to your environment variables:  ")
print(f"  {public_url}")
print(f"=======================================================\n")

## Discord Bot

In [ ]:
# Seraphina Discord API - Local Ollama Framework (Colab Userdata Edition)
# (c) Seraphina Management Team, 2026
# v 1.8 2026-AUG-15

import discord
from discord import app_commands
import aiohttp
import nest_asyncio
import asyncio
import os
from datetime import datetime
import re
from google.colab import userdata

# Apply nest_asyncio to prevent event loop crashes in Colab
nest_asyncio.apply()

# --- FILE I/O HELPERS (LOGS ONLY) ---
def get_log_file_path(guild_name, channel_name, username):
    """Generates and returns the structured filesystem path for log files."""
    server_name = guild_name if guild_name else "Direct_Messages"
    safe_server_name = "".join(c for c in server_name if c.isalnum() or c in " _-")
    safe_channel_name = "".join(c for c in channel_name if c.isalnum() or c in " _-")
    log_folder_path = os.path.join("Logs", safe_server_name)
    os.makedirs(log_folder_path, exist_ok=True)
    return os.path.join(log_folder_path, f"{safe_channel_name}.txt")

def load_recent_history(log_file_path, max_messages=20):
    """Fetches up to the 20 most recent logged entries for context window awareness."""
    if not os.path.exists(log_file_path):
        return ""
    try:
        with open(log_file_path, "r", encoding="utf-8") as file:
            lines = [line.strip() for line in file if line.strip()]
            recent_lines = lines[-max_messages:]
            return "\n".join(recent_lines)
    except Exception as e:
        print(f"[LOG READING ERROR] Failed to fetch history: {e}")
        return ""

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# Load persistent credentials safely via Kaggle Secrets
DISCORD_TOKEN = user_secrets.get_secret('DISCORD_TOKEN')
OLLAMA_ENDPOINT = "http://127.0.0.1:11434/api/chat"

# Setup Discord AutoShardedClient for dynamic scaling and load distribution
intents = discord.Intents.default()
intents.message_content = True
discord_client = discord.AutoShardedClient(intents=intents)
tree = app_commands.CommandTree(discord_client)

# --- LOGGING HELPER ---
def log_message_to_file(username, guild_name, channel_name, content, attachments_list, timestamp):
    """Handles logging to console and dynamic txt files at the final iteration step."""
    server_name = guild_name if guild_name else "Direct_Messages"

    if attachments_list:
        attachments_str = f" : ({' | '.join(attachments_list)})"
    else:
        attachments_str = ""

    log_entry = f"[{username}][{server_name}]: {content}{attachments_str} <[{timestamp}]>"
    log_file_path = get_log_file_path(guild_name, channel_name, username)

    try:
        with open(log_file_path, "a", encoding="utf-8") as log_file:
            log_file.write(log_entry + "\n")
    except Exception as log_error:
        print(f"[LOGGING ERROR] Failed to log message: {log_error}")

    print(log_entry)
    return log_file_path

# --- MESSAGE SPLITTING & MARKDOWN HELPER ---
def split_and_fix_message(text, limit=1900):
    """Splits long messages gracefully and uses Regex to fix broken markdown blocks."""
    chunks = []
    in_triple_block = False
    in_single_block = False

    raw_chunks = []
    while len(text) > limit:
        split_index = text.rfind('\n', 0, limit)
        if split_index == -1:
            split_index = text.rfind(' ', 0, limit)
        if split_index == -1:
            split_index = limit

        raw_chunks.append(text[:split_index])
        text = text[split_index:].lstrip()
    if text:
        raw_chunks.append(text)

    for chunk in raw_chunks:
        if in_triple_block:
            chunk = "```\n" + chunk
        elif in_single_block:
            chunk = "`" + chunk

        triple_matches = len(re.findall(r'```', chunk))
        if triple_matches % 2 != 0:
            in_triple_block = not in_triple_block
            chunk += "\n```"

        temp_chunk = re.sub(r'```', '', chunk)
        single_matches = len(re.findall(r'`', temp_chunk))
        if single_matches % 2 != 0 and not in_triple_block:
            in_single_block = not in_single_block
            chunk += "`"

        chunks.append(chunk)

    return chunks

# --- AI GENERATION HELPER ---
async def generate_seraphina_response(user_text, username, server_name, channel_name, log_file_path, prompt_key="PROMPT"):
    """
    Fetches latest 20 chat history entries, grabs the designated prompt from Kaggle secrets,
    and runs API generation via direct HTTP request to the local Ollama instance.
    """
    system_prompt = user_secrets.get_secret(prompt_key)
    chat_history = await asyncio.to_thread(load_recent_history, log_file_path, 20)

    contextual_prompt = (
        f"{system_prompt}\n\n"
        f"--- CURRENT CONTEXT ---\n"
        f"You are currently talking to user: '{username}'.\n"
        f"Server Name: '{server_name}'\n"
        f"Channel Name: '{channel_name}'\n\n"
        f"Maintain your established persona, instructions, and formatting strictly in your next response."
        f"--- RECENT CHAT HISTORY LOG (Up to 20 messages) ---\n"
        f"{chat_history if chat_history else 'No previous conversation history.'}"
    )

    payload = {
        "model": "Hudson/llama3.1-uncensored:8b",
        "messages": [
            {"role": "system", "content": contextual_prompt},
            {"role": "user", "content": user_text}
        ],
        "stream": False
    }

    try:
        async with aiohttp.ClientSession() as session:
            async with session.post(OLLAMA_ENDPOINT, json=payload) as response:
                if response.status == 200:
                    data = await response.json()
                    return data.get("message", {}).get("content", "")
                else:
                    error_text = await response.text()
                    print(f"[API ERROR] Ollama returned HTTP {response.status}: {error_text}")
                    return "`Action could not be completed. Seraphina couldn't receive your message or she couldn't react to it.`"

    except Exception as api_error:
        print(f"[API ERROR] {api_error}")
        return "`Action could not be completed. Seraphina couldn't receive your message or she couldn't react to it.`"

# --- BACKGROUND RICH PRESENCE ROTATOR ---
async def change_status_loop():
    await discord_client.wait_until_ready()

    phrases = [
        "Feeling bored? my DMs are open~",
        "Aw, you poor cutie~",
        "Need a hug? I'm here~",
        "Ara ara~",
        "Hey now, don't be harsh on yourself~",
        "It's okay, it's gonna be alright~",
        "Observing the quiet..."
    ]

    while not discord_client.is_closed():
        for phrase in phrases:
            try:
                await discord_client.change_presence(
                    status=discord.Status.online,
                    activity=discord.Game(name=phrase)
                )
            except Exception as e:
                print(f"[STATUS ERROR] Failed to update presence: {e}")

            await asyncio.sleep(300)

# --- EVENTS ---
@discord_client.event
async def on_ready():
    await tree.sync()
    discord_client.loop.create_task(change_status_loop())
    print(f'{discord_client.user} is online across {len(discord_client.shards)} shards, slash commands synced.')

@discord_client.event
async def on_message(message):
    if message.author == discord_client.user:
        return

    username = message.author.name
    guild_name = message.guild.name if message.guild else None
    channel_name = getattr(message.channel, 'name', f"DM_with_{username}")
    timestamp = message.created_at.strftime("%Y-%m-%d %H:%M:%S")
    server_name = guild_name if guild_name else "Direct_Messages"
    attachments_list = [att.url for att in message.attachments]

    log_file_path = get_log_file_path(guild_name, channel_name, username)

    if discord_client.user.mentioned_in(message) or isinstance(message.channel, discord.DMChannel):
        async with message.channel.typing():
            user_text = message.clean_content.replace(f'@{discord_client.user.name}', '').strip()

            bot_reply = await generate_seraphina_response(
                user_text, username, server_name, channel_name, log_file_path, prompt_key="PROMPT"
            )

            try:
                message_chunks = split_and_fix_message(bot_reply)
                reply_msg = await message.reply(message_chunks[0], mention_author=False)

                for chunk in message_chunks[1:]:
                    await message.channel.send(chunk)

                reply_timestamp = reply_msg.created_at.strftime("%Y-%m-%d %H:%M:%S")
                await asyncio.to_thread(log_message_to_file, username, guild_name, channel_name, message.clean_content, attachments_list, timestamp)
                await asyncio.to_thread(log_message_to_file, discord_client.user.name, guild_name, channel_name, bot_reply, [], reply_timestamp)

            except Exception as final_error:
                print(f"[CRITICAL ERROR] Completely failed to send message: {final_error}")

# --- SLASH COMMANDS ---
@tree.command(name="chat", description="Converse directly with Seraphina.")
@app_commands.allowed_contexts(guilds=True, dms=True, private_channels=True)
@app_commands.allowed_installs(guilds=True, users=True)
async def chat_command(interaction: discord.Interaction, message: str):
    await interaction.response.defer()

    username = interaction.user.name
    guild_name = interaction.guild.name if interaction.guild else None
    channel_name = getattr(interaction.channel, 'name', f"DM_with_{username}")
    timestamp = interaction.created_at.strftime("%Y-%m-%d %H:%M:%S")
    server_name = guild_name if guild_name else "Direct_Messages"
    log_file_path = get_log_file_path(guild_name, channel_name, username)

    bot_reply = await generate_seraphina_response(
        message, username, server_name, channel_name, log_file_path, prompt_key="PROMPT"
    )

    try:
        message_chunks = split_and_fix_message(bot_reply)
        for chunk in message_chunks:
            await interaction.followup.send(chunk)

        reply_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        await asyncio.to_thread(log_message_to_file, username, guild_name, channel_name, message, [], timestamp)
        await asyncio.to_thread(log_message_to_file, discord_client.user.name, guild_name, channel_name, bot_reply, [], reply_timestamp)
    except Exception as e:
        print(f"[CRITICAL ERROR] /chat command reply failed: {e}")

@tree.command(name="order", description="Converse with Seraphina using the special prompt instructions.")
@app_commands.allowed_contexts(guilds=True, dms=True, private_channels=True)
@app_commands.allowed_installs(guilds=True, users=True)
async def order_command(interaction: discord.Interaction, password: str, message: str):
    if password != "4695":
        await interaction.response.send_message("Access denied. Sorry, you have to use /chat command or just talk to me in my DMs, or @ tag mention me in a server where I have been added/invited.", ephemeral=True)
        return

    await interaction.response.defer()

    username = interaction.user.name
    guild_name = interaction.guild.name if interaction.guild else None
    channel_name = getattr(interaction.channel, 'name', f"DM_with_{username}")
    timestamp = interaction.created_at.strftime("%Y-%m-%d %H:%M:%S")
    server_name = guild_name if guild_name else "Direct_Messages"
    log_file_path = get_log_file_path(guild_name, channel_name, username)

    bot_reply = await generate_seraphina_response(
        message, username, server_name, channel_name, log_file_path, prompt_key="SPECIALPROMPT"
    )

    try:
        message_chunks = split_and_fix_message(bot_reply)
        for chunk in message_chunks:
            await interaction.followup.send(chunk)

        reply_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        await asyncio.to_thread(log_message_to_file, username, guild_name, channel_name, message, [], timestamp)
        await asyncio.to_thread(log_message_to_file, discord_client.user.name, guild_name, channel_name, bot_reply, [], reply_timestamp)
    except Exception as e:
        print(f"[CRITICAL ERROR] /order command reply failed: {e}")

@tree.command(name="forget", description="Delete the current conversation logfile for this context.")
@app_commands.allowed_contexts(guilds=True, dms=True, private_channels=True)
@app_commands.allowed_installs(guilds=True, users=True)
async def forget_command(interaction: discord.Interaction):
    username = interaction.user.name
    guild_name = interaction.guild.name if interaction.guild else None
    channel_name = getattr(interaction.channel, 'name', f"DM_with_{username}")
    log_file_path = get_log_file_path(guild_name, channel_name, username)

    if os.path.exists(log_file_path):
        try:
            await asyncio.to_thread(os.remove, log_file_path)
            await interaction.response.send_message("Logfile cleared for this conversation.", ephemeral=True)
        except Exception as e:
            await interaction.response.send_message(f"Failed to delete logfile: {e}", ephemeral=True)
    else:
        await interaction.response.send_message("No logfile found to delete in this chat.", ephemeral=True)

# --- START THE BOT ---
if __name__ == "__main__":
    if DISCORD_TOKEN:
        try:
            discord_client.run(DISCORD_TOKEN)
        except Exception as e:
            print(f"[CRITICAL ERROR] Failed to connect to Discord: {e}")
    else:
        print("[CRITICAL ERROR] Missing Discord Token. Ensure it is set in Colab userdata.")

# !tail -f ollama.log
"""
import time

def watch_logs(filename="ollama.log"):
    print(f"Listening to {filename} for incoming prompts and responses...\n" + "-"*50)
    with open(filename, 'r') as file:
        # Move the pointer to the end of the file so we only see new traffic
        file.seek(0, 2)
        try:
            while True:
                line = file.readline()
                if not line:
                    time.sleep(0.5) # Wait briefly for new data
                    continue

                # Optional: Filter for lines that contain request/response data to reduce noise
                if "{" in line or "}" in line or "msg=" in line:
                    print(line.strip())
        except KeyboardInterrupt:
            print("\nStopped monitoring.")

watch_logs()
"""